# Week 1: Searching for Interaction Terms (Insurance Fraud, 1,000 rows)
An **interaction term** is two columns multiplied together. It lets the model say "the effect of A depends on B."

This notebook gives you **tools to test ideas**. It does not tell you which interactions to use.

Put `insurance_claims.csv` in the same folder and run top to bottom.

In [38]:
!pip install scikit-learn

## Step 1: Load, clean, and set aside a test set
All searching happens on the **training** data only. The test set stays locked until the very end, so that your final result is honest.

In [39]:
import pandas as pd
import numpy as np
from itertools import combinations
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("insurance_claims.csv").drop(columns=["_c39"])
df = df.replace("?", "Unknown").fillna("Unknown")
df["fraud"] = (df["fraud_reported"] == "Y").astype(int)
df = df.drop(columns=["fraud_reported", "policy_number", "insured_zip",
                      "incident_location", "policy_bind_date", "incident_date"])

X = pd.get_dummies(df.drop(columns=["fraud"]), drop_first=True).astype(float)
y = df["fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

num_cols = list(df.drop(columns=["fraud"]).select_dtypes(include="number").columns)
print("Numeric columns:", len(num_cols))
print("Training rows:", len(X_train))

Numeric columns: 16
Training rows: 800


## Step 2: Tool A - Look at fraud rates for two categories together
Pick two category columns. The table shows the fraud rate and the number of claims in each combination.

**What to look for:** a combination whose fraud rate is very different from what each column suggests alone. **Ignore cells with only a few claims** (small counts are just noise).

In [40]:
train_df = df.loc[X_train.index]

def fraud_table(col1, col2, min_count=15):
    rate = train_df.pivot_table(index=col1, columns=col2, values="fraud", aggfunc="mean").round(2)
    count = train_df.pivot_table(index=col1, columns=col2, values="fraud", aggfunc="count")
    rate = rate.where(count >= min_count)   # hide cells with too few claims
    print("Fraud rate (blank = fewer than", min_count, "claims)")
    display(rate)
    print("Number of claims")
    display(count)

# Try your own pairs, for example:
fraud_table("insured_hobbies", "incident_severity")
#print(sorted(df.select_dtypes(exclude="number").columns))

Fraud rate (blank = fewer than 15 claims)


incident_severity,Major Damage,Minor Damage,Total Loss,Trivial Damage
insured_hobbies,,,,
base-jumping,NaN,NaN,NaN,NaN
basketball,NaN,NaN,NaN,NaN
board-games,NaN,NaN,NaN,NaN
bungie-jumping,NaN,0.06,NaN,NaN
camping,NaN,NaN,NaN,NaN
chess,NaN,NaN,NaN,NaN
cross-fit,NaN,NaN,NaN,NaN
dancing,NaN,0.00,NaN,NaN
exercise,NaN,0.00,NaN,NaN


Number of claims


incident_severity,Major Damage,Minor Damage,Total Loss,Trivial Damage
insured_hobbies,,,,
base-jumping,13,12,11,4
basketball,7,6,11,1
board-games,10,10,10,5
bungie-jumping,11,17,11,5
camping,10,13,11,5
chess,8,13,14,3
cross-fit,6,7,13,2
dancing,7,16,12,4
exercise,14,15,11,2


## Step 3: Tool B - Does a numeric column behave differently across groups?
Splits a numeric column into low / middle / high, then shows the fraud rate by group.

In [41]:
train_df.loc[(train_df["insured_hobbies"]=="video-games") & (train_df["incident_severity"]=="Minor Damage"), "fraud"].agg(["mean","count"])

mean      0.0
count    15.0
Name: fraud, dtype: float64

## Step 4: Tool C - Test ONE interaction with cross-validation
We compare a baseline logistic regression to the same model **plus one interaction**. The score is **ROC AUC** (not accuracy, since 75% of claims are not fraud). We repeat 5-fold cross-validation 3 times so the result is steadier.

**How to read it:** the gain only matters if it is bigger than the noise (the `std` column).

In [42]:
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=1)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))

def cv_auc(features):
    scores = cross_val_score(model, features, y_train, cv=cv, scoring="roc_auc")
    return scores.mean(), scores.std()

base_mean, base_std = cv_auc(X_train)
print(f"Baseline AUC: {base_mean:.3f}  (std {base_std:.3f})")

def test_interaction(a, b):
    X_new = X_train.copy()
    X_new[f"{a} x {b}"] = X_train[a] * X_train[b]
    mean, std = cv_auc(X_new)
    print(f"{a} x {b}: AUC {mean:.3f}  gain {mean - base_mean:+.4f}  (std {std:.3f})")

# Try your own, using any two column names from X_train, for example:
# test_interaction("age", "months_as_customer")

Baseline AUC: 0.831  (std 0.035)


## Step 5: Tool D - Screen ALL pairs of numeric columns
This tests every pair automatically and ranks them by AUC gain.

**Warning:** with many pairs, a few will look good **by chance**. Treat the top of this list as ideas to double-check, not as results.

In [43]:
results = []
for a, b in combinations(num_cols, 2):
    X_new = X_train.copy()
    X_new["inter"] = X_train[a] * X_train[b]
    mean, std = cv_auc(X_new)
    results.append((a, b, mean - base_mean))

screen = pd.DataFrame(results, columns=["col_a", "col_b", "auc_gain"]).sort_values("auc_gain", ascending=False)
print("Pairs tested:", len(screen))
screen.head(10)

Pairs tested: 120


,col_a,col_b,auc_gain
75,capital-loss,incident_hour_of_the_day,0.006846
7,months_as_customer,number_of_vehicles_involved,0.002741
40,policy_deductable,vehicle_claim,0.002477
37,policy_deductable,total_claim_amount,0.001987
94,number_of_vehicles_involved,total_claim_amount,0.001645
96,number_of_vehicles_involved,property_claim,0.001500
12,months_as_customer,property_claim,0.001375
97,number_of_vehicles_involved,vehicle_claim,0.001349
4,months_as_customer,capital-gains,0.001003
85,incident_hour_of_the_day,bodily_injuries,0.000992


## Step 6: Tool E - Let a small decision tree suggest where to look
A tree splits on one column, then another. If two columns keep appearing **together on the same path**, that hints they work together. It is a clue to test with Tool C, not proof.

In [44]:
from sklearn.tree import DecisionTreeClassifier, export_text

tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=20, random_state=0).fit(X_train, y_train)
print(export_text(tree, feature_names=list(X_train.columns)))

|--- insured_hobbies_chess <= 0.50
|   |--- insured_hobbies_cross-fit <= 0.50
|   |   |--- incident_severity_Minor Damage <= 0.50
|   |   |   |--- class: 0
|   |   |--- incident_severity_Minor Damage >  0.50
|   |   |   |--- class: 0
|   |--- insured_hobbies_cross-fit >  0.50
|   |   |--- class: 1
|--- insured_hobbies_chess >  0.50
|   |--- class: 1



## Step 7: Double-check your best ideas (guard against luck)
Re-test your favorite candidates with **different random splits**. A real interaction keeps helping; a lucky one fades.

In [45]:
def recheck(a, b, seeds=(11, 22, 33)):
    gains = []
    for s in seeds:
        cv2 = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=s)
        base = cross_val_score(model, X_train, y_train, cv=cv2, scoring="roc_auc").mean()
        X_new = X_train.copy()
        X_new["inter"] = X_train[a] * X_train[b]
        new = cross_val_score(model, X_new, y_train, cv=cv2, scoring="roc_auc").mean()
        gains.append(new - base)
    print(f"{a} x {b}: gains across seeds = {np.round(gains, 4)}")

# recheck("column_a", "column_b")

## Step 8: Final check on the locked test set (do this ONCE)
After you have chosen your interactions using only training data, compare the model with and without them on the test set.

In [46]:
from sklearn.metrics import roc_auc_score

def final_test(pairs):
    Xtr, Xte = X_train.copy(), X_test.copy()
    for a, b in pairs:
        Xtr[f"{a} x {b}"] = X_train[a] * X_train[b]
        Xte[f"{a} x {b}"] = X_test[a] * X_test[b]
    base = model.fit(X_train, y_train)
    auc_base = roc_auc_score(y_test, base.predict_proba(X_test)[:, 1])
    withint = model.fit(Xtr, y_train)
    auc_int = roc_auc_score(y_test, withint.predict_proba(Xte)[:, 1])
    print(f"Test AUC without interactions: {auc_base:.3f}")
    print(f"Test AUC with interactions:    {auc_int:.3f}")

# final_test([("column_a", "column_b")])

In [47]:
test_interaction("capital-loss", "incident_hour_of_the_day")

capital-loss x incident_hour_of_the_day: AUC 0.838  gain +0.0068  (std 0.035)


In [48]:
recheck("capital-loss", "incident_hour_of_the_day")

capital-loss x incident_hour_of_the_day: gains across seeds = [0.0089 0.0078 0.0074]
